# Propagate and aggregate empirical uncertainty

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from os import environ

configured_root = environ.get("ML4GM_ROOT")
ROOT = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT / "01_Dataset.ipynb").is_file():
        ROOT = ROOT.parent
    if not (ROOT / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set ML4GM_ROOT.")
elif not ROOT.is_dir():
    raise FileNotFoundError(f"The configured ML4GM_ROOT directory does not exist: {ROOT}")
RECON_DIR = ROOT / "results" / "reconstruction"
MODEL_DIR = ROOT / "models"
UNC_DIR = MODEL_DIR / "uncertainty"
UNC_DIR.mkdir(parents=True, exist_ok=True)

# Observations are reference values; measurement uncertainty is not quantified.
# Zero denotes no observation contribution to this reconstruction envelope.
OBS_SIGMA = 0.0
RHO_FALLBACK = 0.25
WINDOW_SIGMA_FLOORS = {
    "reported_or_high_confidence": 0.0,
    "medium_confidence_history": 0.10,
    "nearest_neighbor": 0.20,
    "regional_ensemble": 0.25,
}
Z95 = 1.96

PRODUCTS = {
    "annual": RECON_DIR / "annual_reconstruction_product_1979_2025.csv",
    "winter": RECON_DIR / "winter_reconstruction_product_1979_2025.csv",
    "summer": RECON_DIR / "summer_reconstruction_product_1979_2025.csv",
}
VALIDATION = {
    "annual": MODEL_DIR / "annual" / "xgboost_annual_validation_summary.csv",
    "winter": MODEL_DIR / "annual_seasonal" / "xgboost_winter_validation_summary.csv",
    "summer": MODEL_DIR / "annual_seasonal" / "xgboost_summer_validation_summary.csv",
}
RGI_REGION_SUPPORT = UNC_DIR / "rgi_region_support_for_uncertainty.csv"
EXCLUDED_GLOBAL_REGIONS = [19]


def read_csv(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, low_memory=False)


def rmse(resid):
    resid = pd.to_numeric(pd.Series(resid), errors="coerce").dropna()
    if resid.empty:
        return np.nan
    return float(np.sqrt(np.mean(np.square(resid))))


def weighted_mean(values, weights):
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    w = pd.to_numeric(pd.Series(weights), errors="coerce")
    mask = v.notna() & w.notna() & (w > 0)
    if not mask.any():
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def weighted_std(values, weights):
    v = pd.to_numeric(pd.Series(values), errors="coerce")
    w = pd.to_numeric(pd.Series(weights), errors="coerce")
    mask = v.notna() & w.notna() & (w > 0)
    if mask.sum() < 2:
        return np.nan
    vv = v[mask].to_numpy(dtype=float)
    ww = w[mask].to_numpy(dtype=float)
    mu = np.average(vv, weights=ww)
    return float(np.sqrt(np.average((vv - mu) ** 2, weights=ww)))


def effective_n(weights):
    w = pd.to_numeric(pd.Series(weights), errors="coerce")
    w = w[w.notna() & (w > 0)]
    if w.empty:
        return np.nan
    return float((w.sum() ** 2) / np.square(w).sum())


def validation_base_sigma(experiment):
    df = read_csv(VALIDATION[experiment])
    split_col = "split" if "split" in df.columns else df.columns[0]
    robust = df[df[split_col].astype(str).str.lower().isin(["logo_rgiid", "loyo_year"])]
    if robust.empty:
        robust = df[df[split_col].astype(str).str.contains("logo|loyo", case=False, na=False)]
    if robust.empty:
        robust = df
    return float(pd.to_numeric(robust["RMSE"], errors="coerce").max())


def window_sigma(df, prefix):
    n = len(df)
    out = np.zeros(n, dtype=float)
    confidence_col = f"window_confidence_{prefix}" if f"window_confidence_{prefix}" in df.columns else "window_confidence"
    source_col = f"season_window_source_{prefix}" if f"season_window_source_{prefix}" in df.columns else "season_window_source"
    spread_col = f"{prefix}_pred_std"
    conf = df[confidence_col].astype(str).str.lower() if confidence_col in df.columns else pd.Series([""] * n, index=df.index)
    source = df[source_col].astype(str).str.lower() if source_col in df.columns else pd.Series([""] * n, index=df.index)
    spread = pd.to_numeric(df[spread_col], errors="coerce").fillna(0).to_numpy(dtype=float) if spread_col in df.columns else np.zeros(n, dtype=float)
    out = np.where(conf.eq("medium"), np.maximum(out, WINDOW_SIGMA_FLOORS["medium_confidence_history"]), out)
    out = np.where(conf.isin(["region_fallback", "nearest_neighbor", "low"]), np.maximum(out, WINDOW_SIGMA_FLOORS["nearest_neighbor"]), out)
    out = np.where(source.eq("o1region_ensemble"), np.maximum(spread, WINDOW_SIGMA_FLOORS["regional_ensemble"]), out)
    out = np.maximum(out, spread)
    return out


def uncertainty_class(source_flag, total_sigma, q33, q67):
    source = "" if pd.isna(source_flag) else str(source_flag).lower()
    if "observ" in source:
        return "observed"
    if not np.isfinite(total_sigma):
        return "unsupported_or_fallback"
    if "fallback" in source or "low" in source:
        return "unsupported_or_fallback"
    if total_sigma <= q33:
        return "low_uncertainty"
    if total_sigma <= q67:
        return "medium_uncertainty"
    return "high_uncertainty"


def add_base_uncertainty(df, experiment, base_sigma):
    prefix = experiment
    df = df.copy()
    df[f"{prefix}_model_sigma"] = base_sigma
    df[f"{prefix}_obs_sigma"] = OBS_SIGMA
    if prefix in ["winter", "summer"]:
        df[f"{prefix}_window_sigma"] = window_sigma(df, prefix)
        df[f"{prefix}_method_sigma"] = 0.0
    else:
        df[f"{prefix}_window_sigma"] = 0.0
        direct = pd.to_numeric(df.get("annual_model_pred"), errors="coerce")
        seasonal = pd.to_numeric(df.get("annual_from_seasonal_pred"), errors="coerce")
        df[f"{prefix}_direct_minus_seasonal"] = direct - seasonal
        df[f"{prefix}_method_sigma"] = np.where(
            direct.notna() & seasonal.notna(),
            np.minimum(np.abs(direct - seasonal).to_numpy(dtype=float) / 2.0, base_sigma),
            0.0,
        )
    return df


def finalize_seasonal_uncertainty(df, prefix):
    obs_col = f"{prefix}_obs"
    recon_col = f"{prefix}_recon"
    source_col = f"{prefix}_source_flag"
    model_sigma = df[f"{prefix}_model_sigma"]
    window_std = df[f"{prefix}_window_sigma"]
    has_prediction = df[f"{prefix}_pred"].notna()
    prediction_sigma = np.sqrt(np.square(model_sigma) + np.square(window_std)).where(has_prediction)
    df[f"{prefix}_prediction_sigma"] = prediction_sigma
    is_observed = pd.to_numeric(df[obs_col], errors="coerce").notna() if obs_col in df.columns else pd.Series(False, index=df.index)
    if source_col in df.columns:
        is_observed = is_observed | df[source_col].astype(str).str.lower().str.contains("observ", na=False)
    df[f"{prefix}_total_sigma"] = np.where(is_observed, OBS_SIGMA, prediction_sigma)
    recon = pd.to_numeric(df[recon_col], errors="coerce")
    sigma = pd.to_numeric(df[f"{prefix}_total_sigma"], errors="coerce")
    df[f"{prefix}_lower95"] = recon - Z95 * sigma
    df[f"{prefix}_upper95"] = recon + Z95 * sigma
    df[f"{prefix}_uncertainty_source"] = np.where(is_observed, "observed", "validation_rmse_window_method")
    model_mask = ~is_observed & sigma.notna()
    q33, q67 = sigma[model_mask].quantile([0.33, 0.67]).to_list() if model_mask.any() else (np.nan, np.nan)
    src = df[source_col] if source_col in df.columns else pd.Series([""] * len(df), index=df.index)
    df[f"{prefix}_uncertainty_class"] = [uncertainty_class(s, t, q33, q67) for s, t in zip(src, sigma)]
    return df


def finalize_annual_uncertainty(annual, winter_u, summer_u):
    annual = add_base_uncertainty(annual, "annual", base_sigma["annual"])
    keys = ["RGIId", "year"]
    wcols = keys + ["winter_total_sigma", "winter_window_sigma", "winter_model_sigma", "winter_prediction_sigma"]
    scols = keys + ["summer_total_sigma", "summer_window_sigma", "summer_model_sigma", "summer_prediction_sigma"]
    annual = annual.merge(winter_u[wcols], on=keys, how="left", validate="one_to_one")
    annual = annual.merge(summer_u[scols], on=keys, how="left", validate="one_to_one")
    w = annual["winter_prediction_sigma"]
    s = annual["summer_prediction_sigma"]
    annual["annual_from_seasonal_sigma"] = np.sqrt(np.square(w) + np.square(s) + 2 * RHO_FALLBACK * w * s)
    direct_sigma = np.sqrt(
        np.square(pd.to_numeric(annual["annual_model_sigma"], errors="coerce"))
        + np.square(pd.to_numeric(annual["annual_window_sigma"], errors="coerce"))
        + np.square(pd.to_numeric(annual["annual_method_sigma"], errors="coerce"))
    )
    use_seasonal = annual["annual_method_flag"].eq("seasonal_sum")
    seasonal_sigma = np.sqrt(
        np.square(pd.to_numeric(annual["annual_from_seasonal_sigma"], errors="coerce"))
        + np.square(pd.to_numeric(annual["annual_method_sigma"], errors="coerce"))
    )
    has_seasonal_prediction = annual["annual_from_seasonal_pred"].notna()
    if ((has_seasonal_prediction | use_seasonal) & seasonal_sigma.isna()).any():
        raise ValueError("Winter and summer prediction uncertainties are required for the seasonal-sum prediction.")
    direct_prediction_sigma = direct_sigma.where(annual["annual_model_pred"].notna())
    annual["annual_prediction_sigma"] = seasonal_sigma.where(has_seasonal_prediction, direct_prediction_sigma)
    annual["annual_total_sigma"] = seasonal_sigma.where(use_seasonal, direct_sigma)
    annual["annual_obs_sigma"] = OBS_SIGMA
    annual["annual_sigma_method_flag"] = np.where(use_seasonal, "seasonal_sum", "direct_annual")
    obs = pd.to_numeric(annual.get("annual_obs"), errors="coerce").notna()
    if "annual_source_flag" in annual.columns:
        obs = obs | annual["annual_source_flag"].astype(str).str.lower().str.contains("observ", na=False)
    annual["annual_total_sigma"] = np.where(obs, OBS_SIGMA, annual["annual_total_sigma"])
    recon = pd.to_numeric(annual["annual_recon"], errors="coerce")
    sigma = pd.to_numeric(annual["annual_total_sigma"], errors="coerce")
    annual["annual_lower95"] = recon - Z95 * sigma
    annual["annual_upper95"] = recon + Z95 * sigma
    annual["annual_uncertainty_source"] = np.where(obs, "observed", "validation_rmse_window_method")
    model_mask = ~obs & sigma.notna()
    q33, q67 = sigma[model_mask].quantile([0.33, 0.67]).to_list() if model_mask.any() else (np.nan, np.nan)
    src = annual["annual_source_flag"] if "annual_source_flag" in annual.columns else pd.Series([""] * len(annual), index=annual.index)
    annual["annual_uncertainty_class"] = [uncertainty_class(sv, tv, q33, q67) for sv, tv in zip(src, sigma)]
    return annual


def summarize_regional_uncertainty(df, experiment):
    prefix = experiment
    value_col = f"{prefix}_recon"
    sigma_col = f"{prefix}_total_sigma"
    rows = []
    for (region, name, year), g in df.groupby(["O1Region", "region_name", "year"], dropna=False):
        weights = pd.to_numeric(g["Area"], errors="coerce").fillna(0)
        total_area = float(weights.sum())
        if total_area <= 0:
            continue
        vals = pd.to_numeric(g[value_col], errors="coerce")
        sig = pd.to_numeric(g[sigma_col], errors="coerce")
        mask = vals.notna() & sig.notna() & (weights > 0)
        if not mask.any():
            continue
        ww = weights[mask] / weights[mask].sum()
        model_se = float(np.sqrt(np.sum(np.square(ww.to_numpy(dtype=float)) * np.square(sig[mask].to_numpy(dtype=float)))))
        n_eff = effective_n(weights[mask])
        spread = weighted_std(vals[mask], weights[mask])
        spread_se = float(spread / np.sqrt(n_eff)) if np.isfinite(spread) and np.isfinite(n_eff) and n_eff > 0 else 0.0
        regional_sigma = float(np.sqrt(model_se ** 2 + spread_se ** 2))
        mean = weighted_mean(vals[mask], weights[mask])
        rows.append({
            "O1Region": region,
            "region_name": name,
            "year": int(year),
            "area_weighted_mean_pred": mean,
            "regional_sigma": regional_sigma,
            "lower95": mean - Z95 * regional_sigma,
            "upper95": mean + Z95 * regional_sigma,
            "model_se": model_se,
            "spread_se": spread_se,
            "n_glaciers": int(g["RGIId"].nunique()),
            "sum_area": total_area,
            "n_eff": n_eff,
            "experiment": experiment,
        })
    return pd.DataFrame(rows).sort_values(["O1Region", "year"])


def load_rgi_region_weights(regional_by_exp):
    if not RGI_REGION_SUPPORT.is_file():
        raise FileNotFoundError(f"Run 01_Dataset.ipynb to derive RGI glacier-area weights: {RGI_REGION_SUPPORT}")
    support = read_csv(RGI_REGION_SUPPORT)
    support["O1Region"] = pd.to_numeric(support["O1Region"], errors="raise")
    support["rgi_area"] = pd.to_numeric(support["rgi_area"], errors="raise")
    if (set(support["O1Region"]) != set(range(1, 20)) or support["O1Region"].duplicated().any()
            or not np.isfinite(support["rgi_area"]).all() or not support["rgi_area"].gt(0).all()):
        raise ValueError("Expected positive glacier areas for the 19 RGI 6.0 first-order regions.")
    supported = set(support["O1Region"])
    for regional in regional_by_exp.values():
        missing = set(regional["O1Region"]) - supported
        if missing:
            raise ValueError(f"Missing RGI entries for product regions: {sorted(missing)}")
    return support.loc[~support["O1Region"].isin(EXCLUDED_GLOBAL_REGIONS), ["O1Region", "rgi_area"]]


def global_timeseries(regional_by_exp):
    rgi_weights = load_rgi_region_weights(regional_by_exp)
    frames = []
    for exp, reg in regional_by_exp.items():
        reg = reg.merge(rgi_weights, on="O1Region", how="left")
        for year, g in reg.groupby("year"):
            weights = pd.to_numeric(g["rgi_area"], errors="coerce")
            vals = pd.to_numeric(g["area_weighted_mean_pred"], errors="coerce")
            sig = pd.to_numeric(g["regional_sigma"], errors="coerce")
            mask = weights.notna() & vals.notna() & sig.notna() & (weights > 0)
            if not mask.any():
                continue
            ww = weights[mask] / weights[mask].sum()
            mean = float(np.average(vals[mask], weights=weights[mask]))
            sigma = float(np.sqrt(np.sum(np.square(ww.to_numpy(dtype=float)) * np.square(sig[mask].to_numpy(dtype=float)))))
            frames.append({
                "year": int(year),
                "experiment": exp,
                f"{exp}_mean": mean,
                f"{exp}_sigma": sigma,
                f"{exp}_lower95": mean - Z95 * sigma,
                f"{exp}_upper95": mean + Z95 * sigma,
                f"{exp}_n_regions": int(mask.sum()),
            })
    long = pd.DataFrame(frames)
    if long.empty:
        return long
    out = pd.DataFrame({"year": sorted(long["year"].unique())})
    for exp in regional_by_exp:
        exp_cols = [
            "year",
            f"{exp}_mean",
            f"{exp}_sigma",
            f"{exp}_lower95",
            f"{exp}_upper95",
            f"{exp}_n_regions",
        ]
        sub = long.loc[long["experiment"].eq(exp), exp_cols].copy()
        out = out.merge(sub, on="year", how="left")
    return out.sort_values("year")


def calibration_summary(products):
    rows = []
    pred_candidates = {
        "annual": ["annual_consensus_pred", "annual_from_seasonal_pred", "annual_model_pred"],
        "winter": ["winter_pred", "winter_global_pred"],
        "summer": ["summer_pred", "summer_global_pred"],
    }
    for experiment, df in products.items():
        prefix = experiment
        obs_col = f"{prefix}_obs"
        pred_col = next((c for c in pred_candidates[experiment] if c in df.columns), f"{prefix}_recon")
        class_col = f"{prefix}_uncertainty_class"
        source_col = f"{prefix}_source_flag"
        eval_sigma = df[f"{prefix}_prediction_sigma"]
        resid = pd.to_numeric(df[obs_col], errors="coerce") - pd.to_numeric(df[pred_col], errors="coerce")
        mask = resid.notna() & eval_sigma.notna() & (eval_sigma > 0)

        def add_row(group_type, group, gmask):
            r = resid[gmask]
            s = eval_sigma[gmask]
            z = r / s
            rows.append({
                "experiment": experiment,
                "prediction_column": pred_col,
                "group_type": group_type,
                "group": group,
                "n": int(gmask.sum()),
                "rmse": rmse(r),
                "mean_sigma": float(s.mean()) if len(s) else np.nan,
                "median_sigma": float(s.median()) if len(s) else np.nan,
                "coverage_68": float((np.abs(z) <= 1.0).mean()) if len(z) else np.nan,
                "coverage_95": float((np.abs(z) <= Z95).mean()) if len(z) else np.nan,
                "mean_z": float(z.mean()) if len(z) else np.nan,
                "std_z": float(z.std(ddof=0)) if len(z) else np.nan,
            })

        add_row("all", "all", mask)
        if class_col in df.columns:
            for group, idx in df.groupby(class_col).groups.items():
                add_row("uncertainty_class", group, mask & df.index.isin(idx))
        if source_col in df.columns:
            for group, idx in df.groupby(source_col).groups.items():
                add_row("source_flag", group, mask & df.index.isin(idx))
        if "O1Region" in df.columns:
            for group, idx in df.groupby("O1Region").groups.items():
                add_row("O1Region", group, mask & df.index.isin(idx))
    return pd.DataFrame(rows)


# Load inputs and calibrate base uncertainty.
base_sigma = {exp: validation_base_sigma(exp) for exp in PRODUCTS}
annual = read_csv(PRODUCTS["annual"])
winter = read_csv(PRODUCTS["winter"])
summer = read_csv(PRODUCTS["summer"])

winter_u = finalize_seasonal_uncertainty(add_base_uncertainty(winter, "winter", base_sigma["winter"]), "winter")
summer_u = finalize_seasonal_uncertainty(add_base_uncertainty(summer, "summer", base_sigma["summer"]), "summer")
annual_u = finalize_annual_uncertainty(annual, winter_u, summer_u)

winter_u.to_csv(RECON_DIR / "winter_reconstruction_product_1979_2025_uncertainty.csv", index=False)
summer_u.to_csv(RECON_DIR / "summer_reconstruction_product_1979_2025_uncertainty.csv", index=False)
annual_u.to_csv(RECON_DIR / "annual_reconstruction_product_1979_2025_uncertainty.csv", index=False)

regional_u = {
    "annual": summarize_regional_uncertainty(annual_u, "annual"),
    "winter": summarize_regional_uncertainty(winter_u, "winter"),
    "summer": summarize_regional_uncertainty(summer_u, "summer"),
}
for exp, df in regional_u.items():
    df.to_csv(RECON_DIR / f"regional_{exp}_balance_summary_uncertainty.csv", index=False)

global_u = global_timeseries(regional_u)
global_u.to_csv(RECON_DIR / "global_timeseries_uncertainty.csv", index=False)

calib = calibration_summary({"annual": annual_u, "winter": winter_u, "summer": summer_u})
calib.to_csv(UNC_DIR / "uncertainty_calibration_summary.csv", index=False)

metadata = {
    "obs_sigma": OBS_SIGMA,
    "observation_uncertainty_policy": "Observations are reference values; measurement uncertainty is not quantified and contributes zero to the reconstruction envelope.",
    "annual_seasonal_sum_policy": "The covariance term uses the winter and summer model prediction standard uncertainties, including where a seasonal observation is retained.",
    "rho_fallback": RHO_FALLBACK,
    "window_sigma_floors": WINDOW_SIGMA_FLOORS,
    "base_sigma": base_sigma,
    "components": ["model_sigma", "window_sigma", "method_sigma"],
    "global_timeseries_weighting": "RGI O1Region area among supported regions, excluding O1Region 19",
    "excluded_global_regions": EXCLUDED_GLOBAL_REGIONS,
    "regional_bias_correction_removed": True,
    "outputs": {
        "annual": (RECON_DIR / "annual_reconstruction_product_1979_2025_uncertainty.csv").relative_to(ROOT).as_posix(),
        "winter": (RECON_DIR / "winter_reconstruction_product_1979_2025_uncertainty.csv").relative_to(ROOT).as_posix(),
        "summer": (RECON_DIR / "summer_reconstruction_product_1979_2025_uncertainty.csv").relative_to(ROOT).as_posix(),
        "global_timeseries": (RECON_DIR / "global_timeseries_uncertainty.csv").relative_to(ROOT).as_posix(),
    },
}
(UNC_DIR / "uncertainty_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# Basic validation checks.
for exp, df in {"annual": annual_u, "winter": winter_u, "summer": summer_u}.items():
    sig = pd.to_numeric(df[f"{exp}_total_sigma"], errors="coerce")
    lo = pd.to_numeric(df[f"{exp}_lower95"], errors="coerce")
    hi = pd.to_numeric(df[f"{exp}_upper95"], errors="coerce")
    if sig.isna().any() or (sig < 0).any():
        raise ValueError(f"Invalid sigma values in {exp}")
    if (lo > hi).any():
        raise ValueError(f"Invalid 95% bounds in {exp}")

print("Base sigma:", base_sigma)
print("Annual rows:", len(annual_u), "Winter rows:", len(winter_u), "Summer rows:", len(summer_u))
print("Global uncertainty file:", RECON_DIR / "global_timeseries_uncertainty.csv")